In [ ]:
! pip show langchain

In [2]:
! python --version

133.86s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Python 3.13.3


In [17]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  

llm = ChatOpenAI(
    model="mimo-v2.5-pro", 
    temperature=0.7,
    timeout=30,
    max_tokens=200,
    stop="我",
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

llm.invoke("介绍一下你自己")



AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 253, 'total_tokens': 258, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 4, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}}, 'model_provider': 'openai', 'model_name': 'mimo-v2.5-pro', 'system_fingerprint': None, 'id': '21e50a11fcac49fc9e11bcb02b7eb1cc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4324-4735-7b02-ade7-74947462ce7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 253, 'output_tokens': 5, 'total_tokens': 258, 'input_token_details': {'cache_read': 192}, 'output_token_details': {'reasoning': 4}})

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  

llm = ChatOpenAI(
    model="mimo-v2.5-pro", 
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

question = "langchain 是什么？"
# llm.invoke(question)

# for chunk in llm.stream(question):
#     print(chunk.content + "|")

# llm.batch([question, "langchain 作者是谁"])


async for event in llm.astream_events(question, version="v2"):
  print(f"event: {event} | name: {event['name']} | data: {event['data']}")


In [26]:
import os
from typing import Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)


class Joke(BaseModel):
    """Joke to tell user"""
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(default=None, description="How funny the joke is, from 1 to 10")


# MiMo 默认 structured output 会混入 reasoning 文本，导致 JSON 解析失败
structured_llm = llm.with_structured_output(Joke, method="function_calling")
structured_llm.invoke("给我讲一个程序员的笑话")

Joke(setup='为什么程序员总是把万圣节和圣诞节搞混？', punchline='因为 Oct 31 = Dec 25 ！（八进制的31等于十进制的25）', rating=7)

In [27]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

res = llm.invoke("介绍一下你自己")
res.usage_metadata

{'input_tokens': 253,
 'output_tokens': 283,
 'total_tokens': 536,
 'input_token_details': {'cache_read': 192},
 'output_token_details': {'reasoning': 186}}

In [29]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
  """Get the product of two numbers"""
  return a * b



print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Get the product of two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
prompt = PromptTemplate.from_template("你是一个{name}, 帮我起一个具有{country}特色的{sex}名字")
prompts = prompt.format(name="小明", country="中国", sex="男")
print(prompts)


chat_template = ChatPromptTemplate.from_messages(
  [
    ("system", "你是一个{name}, 帮我起一个具有{country}特色的{sex}名字"),
    ("user", "{input}"),
  ]
)

chat_templates = chat_template.format(name="小明", country="中国", sex="男", input="帮我起一个名字")
print(chat_templates)


你是一个小明, 帮我起一个具有中国特色的男名字
System: 你是一个小明, 帮我起一个具有中国特色的男名字
Human: 帮我起一个名字


In [7]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import MessagesPlaceholder


prompt_template = ChatPromptTemplate([
  ("system", "你是一个厉害的 AI 人工智能助手"),
  MessagesPlaceholder("msg")
])

result = prompt_template.invoke({"msg": [HumanMessage(content="Hi")] })
print(result)



sy = SystemMessage(
  content="你是一个大师",
  additional_kwargs={
    "大师名字": "陈瞎子"
  }
)

hu = HumanMessage(
  content="请问大师叫什么"
)

ai = AIMessage(
  content="我叫陈瞎子"
)

[sy, hu, ai]






messages=[SystemMessage(content='你是一个厉害的 AI 人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={})]


[SystemMessage(content='你是一个大师', additional_kwargs={'大师名字': '陈瞎子'}, response_metadata={}),
 HumanMessage(content='请问大师叫什么', additional_kwargs={}, response_metadata={}),
 AIMessage(content='我叫陈瞎子', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
import inspect

from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, StringPromptTemplate


def hello(a):
  print("hello")
  return a

PROMPT = """你是一个非常有经验和天赋的程序员，现在给你如下函数名称，你会按照如下格式，输出这段代码的名称、源代码、中文解释。
函数名称：{function_name}
函数源代码：{source_code}
函数中文解释：{function_chinese_explanation}
"""

def get_source_code(function_name):
  return inspect.getsource(function_name)

class CustomPrompt(StringPromptTemplate):
  def format(self, **kwargs):
    source_code = get_source_code(kwargs["function_name"])
    return PROMPT.format(function_name=kwargs["function_name"].__name__, source_code=source_code, function_chinese_explanation="")


a = CustomPrompt(input_variables=["function_name"])
prompt = a.format(function_name=hello)
# print(prompt)


import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0,
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)


# msg = llm.invoke(prompt)
# print(msg)



example = [
  {"input": "2 🐦‍⬛ 2", "output": "4"},
  {"input": "3 🐦‍⬛ 3", "output": "9"},
]

example_prompt = ChatPromptTemplate.from_messages(
  [
    ("human", "{input}"),
    ("ai", "{output}"),
  ]
)

few_show_prompt = FewShotChatMessagePromptTemplate(
  example_prompt=example_prompt,
  examples=example,
)

print(few_show_prompt.invoke({}).to_messages())



final_prompt = ChatPromptTemplate.from_messages(
  [
    ("system", "你是一个神奇的数学奇才"),
    few_show_prompt,
    ("human", "{input}"),
  ]
)

chain = final_prompt | llm
result = chain.invoke({"input": "What is 2 🐦‍⬛ 9"})
print(result)

[HumanMessage(content='2 🐦\u200d⬛ 2', additional_kwargs={}, response_metadata={}), AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='3 🐦\u200d⬛ 3', additional_kwargs={}, response_metadata={}), AIMessage(content='9', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
content='Looking at the pattern:\n- 2 🐦\u200d⬛ 2 = 4 → 2 × 2 = 4\n- 3 🐦\u200d⬛ 3 = 9 → 3 × 3 = 9\n\nSo 🐦\u200d⬛ represents **multiplication (×)**.\n\n2 🐦\u200d⬛ 9 = 2 × 9 = **18**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 268, 'prompt_tokens': 70, 'total_tokens': 338, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 177, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'mimo-v2.5-pro', 'system_fingerprint':

In [ ]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("{foo}11{bar}")
partial_prompt = prompt.partial(foo="foo")
print(partial_prompt.invoke({"bar": "bar"}))



def get_date():
  now = datetime.now()
  return now.strftime("%Y-%m-%d %H:%M:%S")

prompt = PromptTemplate(
  template="Tell me a {adjective} joke about the date {date}",
  input_variables=["adjective", "date"]
)

partial_prompt = prompt.partial(date=get_date)
print(partial_prompt.invoke({"adjective": "funny"}))


text='foo11bar'
text='Tell me a funny joke about the date 2026-05-21 10:47:37'


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
load_dotenv()


@tool
def get_weather(location: str) -> str:
  """根据 location 名称获取天气"""
  return "今天天气晴朗"



llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0,
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

llm_with_tools = llm.bind_tools([get_weather])

llmChat = llm_with_tools | StrOutputParser()
res = llmChat.invoke("上海今天的天气怎么样")
print(res)

In [8]:
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, model_validator,Field
from langchain_core.prompts import PromptTemplate
load_dotenv()



llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0,
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

class Joke(BaseModel):
  setup: str = Field(description="笑话中的铺垫问题，必须以？结尾")
  punchline: str = Field(description="笑话中回答铺垫问题的部分，通常是一种抖包袱方式回答铺垫间题，例如谐音、会错意等")

  @model_validator(mode="after")
  def question_ends_with_question_mark(self) -> "Joke":
    if self.setup and self.setup[-1] not in ("?", "？"):
      raise ValueError("笑话的铺垫问题必须以？结尾")
    return self

parser = PydanticOutputParser(pydantic_object=Joke)
# parser = JsonOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
  template="回答用户的查询.\n{format_instructions}\n(query 用户查询）\n",
  input_variables=["query"],
  partial_variables={"format_instructions": parser.get_format_instructions()},
)

prompt_chain = prompt | llm | parser
output = prompt_chain.invoke({"query": "给我讲一个程序员的笑话"})


print(parser.get_format_instructions())
print(output)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"setup": {"description": "笑话中的铺垫问题，必须以？结尾", "title": "Setup", "type": "string"}, "punchline": {"description": "笑话中回答铺垫问题的部分，通常是一种抖包袱方式回答铺垫间题，例如谐音、会错意等", "title": "Punchline", "type": "string"}}, "required": ["setup", "punchline"]}
```
setup='为什么书总是在害怕？' punchline='因为它总被“读”（毒）。'
